In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
df = pd.read_csv("student_data.csv")

In [ ]:
df['higher'] = df['higher'].map({
    'yes': 1,
    'no': 0
})


In [ ]:
numerical_features = [
    'age',
    'Medu',
    'Fedu',
    'traveltime',
    'studytime',
    'failures',
    'famrel',
    'freetime',
    'goout',
    'Dalc',
    'Walc',
    'health',
    'absences',
    'G1',
    'G2'
]


In [ ]:
X = df[numerical_features]
y = df['higher']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
scaler = StandardScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train, y_train)
logistic_pred = logistic_model.predict(X_test)

In [ ]:
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)

In [ ]:
k_values = [3, 5, 7]
knn_results = []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    knn_pred = knn.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, knn_pred)
    knn_results.append({
        'k': k,
        'accuracy': accuracy
    })

In [ ]:
knn_results_df = pd.DataFrame(knn_results)
best_k = int(
    knn_results_df.loc[
        knn_results_df['accuracy'].idxmax(),
        'k'
    ]
)

In [ ]:
knn_model = KNeighborsClassifier(
    n_neighbors=best_k
)
knn_model.fit(X_train_scaled, y_train)
knn_pred = knn_model.predict(X_test_scaled)

In [ ]:
models = {
    'Logistic Regression': logistic_pred,
    'Naive Bayes': nb_pred,
    'KNN': knn_pred
}

In [ ]:
results = []
for model_name, predictions in models.items():
    accuracy = accuracy_score(
        y_test,
        predictions
    )
    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )
    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )
    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )
    results.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    })

In [ ]:
results_df = pd.DataFrame(results)

In [ ]:
for model_name, predictions in models.items():
    print("\n")
    print("=" * 60)
    print(model_name)
    print("=" * 60)

    print(
        classification_report(
            y_test,
            predictions,
            target_names=['No', 'Yes'],
            zero_division=0
        )
    )

In [ ]:
for model_name, predictions in models.items():
    cm = confusion_matrix(
        y_test,
        predictions
    )
    print("\n")
    print("=" * 40)
    print(model_name)
    print("=" * 40)
    print(cm)

In [ ]:
for model_name, predictions in models.items():

    cm = confusion_matrix(
        y_test,
        predictions
    )

    plt.figure(figsize=(5, 4))

    plt.imshow(cm)

    plt.title(model_name + " - Confusion Matrix")

    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.xticks(
        [0, 1],
        ['No', 'Yes']
    )

    plt.yticks(
        [0, 1],
        ['No', 'Yes']
    )

    # Display values inside matrix
    for i in range(2):
        for j in range(2):
            plt.text(
                j,
                i,
                cm[i, j],
                ha='center',
                va='center'
            )

    plt.colorbar()